In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cv2
import nibabel as nib

import tensorflow as tf

from scipy.ndimage import binary_fill_holes
from scipy.spatial.distance import cdist

print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPU(s):", gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✓ GPU memory growth enabled")
    except RuntimeError as e:
        print("GPU memory-growth warning:", e)

TensorFlow: 2.20.0
GPU(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ GPU memory growth enabled


In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("✓ TensorFlow deterministic operations enabled")
except Exception as e:
    print("Determinism note:", e)

print("Seed:", SEED)

✓ TensorFlow deterministic operations enabled
Seed: 42


In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/CAMUS_FINAL")

PHASE1_DIR = PROJECT_DIR / "phase1_segmentation"

# Previous completed experiment
P1A_DIR = PHASE1_DIR / "phase1A_resolution"
P1A_CONFIG_DIR = P1A_DIR / "configs"
P1A_TABLE_DIR = P1A_DIR / "tables"
P1A_MODEL_DIR = P1A_DIR / "models"

# New Phase 1B
P1B_DIR = PHASE1_DIR / "phase1B_hyperparameters"

P1B_CONFIG_DIR = P1B_DIR / "configs"
P1B_MODEL_DIR = P1B_DIR / "models"
P1B_HISTORY_DIR = P1B_DIR / "histories"
P1B_TABLE_DIR = P1B_DIR / "tables"
P1B_PLOT_DIR = P1B_DIR / "plots"

for folder in [
    P1B_DIR,
    P1B_CONFIG_DIR,
    P1B_MODEL_DIR,
    P1B_HISTORY_DIR,
    P1B_TABLE_DIR,
    P1B_PLOT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Phase 1A:", P1A_DIR)
print("Phase 1B:", P1B_DIR)
print("\n✓ Phase 1B folder structure ready.")

Project: /content/drive/MyDrive/CAMUS_FINAL
Phase 1A: /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1A_resolution
Phase 1B: /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters

✓ Phase 1B folder structure ready.


In [ ]:
P1A_DECISION_PATH = (
    P1A_CONFIG_DIR
    / "phase1A_final_resolution_decision.json"
)

assert P1A_DECISION_PATH.exists(), (
    f"Phase 1A decision file not found:\n{P1A_DECISION_PATH}"
)

with open(P1A_DECISION_PATH, "r") as f:
    phase1a_decision = json.load(f)

SELECTED_RESOLUTION = int(
    phase1a_decision["selected_resolution"]
)

assert SELECTED_RESOLUTION == 256, (
    "Phase 1B expected the Phase 1A selected resolution to be 256."
)

print("Phase 1A status:",
      phase1a_decision["status"])

print(
    "Selected resolution:",
    f"{SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}"
)

print(
    "Phase 1A parameter count:",
    phase1a_decision["parameter_count"]
)

print("\n✓ Phase 1A decision successfully loaded.")
print("✓ Resolution locked at 256×256 for Phase 1B.")

Phase 1A status: COMPLETED
Selected resolution: 256×256
Phase 1A parameter count: 1940817

✓ Phase 1A decision successfully loaded.
✓ Resolution locked at 256×256 for Phase 1B.


In [ ]:
DATASET_PATH = Path(
    "/content/camus_data_final/database_nifti"
)

if DATASET_PATH.exists():
    print("✓ CAMUS dataset already extracted.")
    print("Dataset:", DATASET_PATH)

else:
    print("CAMUS dataset is not currently extracted.")
    print("The archive will need to be extracted again.")

CAMUS dataset is not currently extracted.
The archive will need to be extracted again.


In [ ]:
import zipfile
import shutil

OUTER_ZIP_PATH = Path(
    "/content/drive/MyDrive/download.zip"
)

OUTER_EXTRACT_DIR = Path(
    "/content/camus_data"
)

INNER_ARCHIVE_PATH = (
    OUTER_EXTRACT_DIR / "download"
)

INNER_EXTRACT_DIR = Path(
    "/content/camus_data_final"
)

DATASET_PATH = (
    INNER_EXTRACT_DIR / "database_nifti"
)


if DATASET_PATH.exists():

    print("✓ Dataset already available.")
    print(DATASET_PATH)

else:

    assert OUTER_ZIP_PATH.exists(), (
        f"Outer archive not found:\n{OUTER_ZIP_PATH}"
    )

    # ------------------------------------------------------------
    # Extract outer archive
    # ------------------------------------------------------------

    if not INNER_ARCHIVE_PATH.exists():

        print("Extracting outer archive...")

        OUTER_EXTRACT_DIR.mkdir(
            parents=True,
            exist_ok=True
        )

        with zipfile.ZipFile(
            OUTER_ZIP_PATH,
            "r"
        ) as outer_zip:

            outer_zip.extractall(
                OUTER_EXTRACT_DIR
            )

        print("✓ Outer archive extracted.")

    else:

        print("✓ Inner archive already available.")


    # ------------------------------------------------------------
    # Extract inner archive
    # ------------------------------------------------------------

    if not DATASET_PATH.exists():

        print("Extracting inner CAMUS archive...")

        INNER_EXTRACT_DIR.mkdir(
            parents=True,
            exist_ok=True
        )

        with zipfile.ZipFile(
            INNER_ARCHIVE_PATH,
            "r"
        ) as inner_zip:

            inner_zip.extractall(
                INNER_EXTRACT_DIR
            )

        print("✓ Inner archive extracted.")


assert DATASET_PATH.exists(), (
    "CAMUS database_nifti folder was not found "
    "after extraction."
)

print("\n✓ CAMUS dataset ready:")
print(DATASET_PATH)

Extracting outer archive...
✓ Outer archive extracted.
Extracting inner CAMUS archive...
✓ Inner archive extracted.

✓ CAMUS dataset ready:
/content/camus_data_final/database_nifti


In [ ]:
split_candidates = list(
    PROJECT_DIR.rglob("patient_split_seed42.json")
)

print("Split files found:")

for path in split_candidates:
    print(" -", path)

assert len(split_candidates) >= 1, (
    "Could not find patient_split_seed42.json "
    "inside CAMUS_FINAL."
)

SPLIT_PATH = split_candidates[0]

with open(SPLIT_PATH, "r") as f:
    patient_split = json.load(f)


# Support the naming used by the saved split
train_ids = patient_split.get(
    "train_ids",
    patient_split.get("train")
)

val_ids = patient_split.get(
    "val_ids",
    patient_split.get("validation",
    patient_split.get("val"))
)

test_ids = patient_split.get(
    "test_ids",
    patient_split.get("test")
)


assert train_ids is not None
assert val_ids is not None
assert test_ids is not None

assert len(train_ids) == 400
assert len(val_ids) == 50
assert len(test_ids) == 50

assert len(
    set(train_ids) &
    set(val_ids)
) == 0

assert len(
    set(train_ids) &
    set(test_ids)
) == 0

assert len(
    set(val_ids) &
    set(test_ids)
) == 0


print("\nSplit file:")
print(SPLIT_PATH)

print("\nTrain patients:", len(train_ids))
print("Validation patients:", len(val_ids))
print("Test patients:", len(test_ids))

print("\nIMPORTANT:")
print("Test patients will NOT be used in Phase 1B.")

print("\n✓ Original patient-level split verified.")

Split files found:
 - /content/drive/MyDrive/CAMUS_FINAL/splits/patient_split_seed42.json

Split file:
/content/drive/MyDrive/CAMUS_FINAL/splits/patient_split_seed42.json

Train patients: 400
Validation patients: 50
Test patients: 50

IMPORTANT:
Test patients will NOT be used in Phase 1B.

✓ Original patient-level split verified.


In [ ]:
baseline_256_path = (
    P1A_MODEL_DIR
    / "baseline_unet_256.keras"
)

assert baseline_256_path.exists(), (
    f"Phase 1A 256 model not found:\n{baseline_256_path}"
)

patient_folders = sorted([
    p.name
    for p in DATASET_PATH.iterdir()
    if p.is_dir()
    and p.name.startswith("patient")
])

assert len(patient_folders) == 500, (
    f"Expected 500 CAMUS patients, "
    f"found {len(patient_folders)}."
)


print("=" * 70)
print("PHASE 1B — PRE-EXPERIMENT VERIFICATION")
print("=" * 70)

print(
    "\nLocked resolution:",
    f"{SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}"
)

print(
    "Baseline architecture parameters:",
    phase1a_decision["parameter_count"]
)

print("Train patients:", len(train_ids))
print("Validation patients:", len(val_ids))
print("Test patients:", len(test_ids))

print(
    "\nPhase 1A baseline checkpoint:"
)
print(baseline_256_path)

print(
    "\nDataset patients verified:",
    len(patient_folders)
)

print("\nPHASE 1B ORDER:")
print("1B-1  Learning rate")
print("1B-2  Optimizer")
print("1B-3  Batch size")
print("1B-4  Augmentation")

print("\nNOT being changed yet:")
print("• Resolution")
print("• U-Net capacity")
print("• Loss function")
print("• Threshold")

print("\nTest set used for model selection: NO")

print("\n✓ Ready to begin Phase 1B-1.")

PHASE 1B — PRE-EXPERIMENT VERIFICATION

Locked resolution: 256×256
Baseline architecture parameters: 1940817
Train patients: 400
Validation patients: 50
Test patients: 50

Phase 1A baseline checkpoint:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1A_resolution/models/baseline_unet_256.keras

Dataset patients verified: 500

PHASE 1B ORDER:
1B-1  Learning rate
1B-2  Optimizer
1B-3  Batch size
1B-4  Augmentation

NOT being changed yet:
• Resolution
• U-Net capacity
• Loss function
• Threshold

Test set used for model selection: NO

✓ Ready to begin Phase 1B-1.


In [ ]:
# ================================================================
# PHASE 1B — CELL 9
# CAMUS FRAME LOADING + PREPROCESSING
# ================================================================

def load_nifti_2d(file_path):
    nii = nib.load(str(file_path))
    data = np.asarray(nii.get_fdata(), dtype=np.float32)

    if data.ndim == 3 and data.shape[-1] == 1:
        data = data[:, :, 0]

    if data.ndim != 2:
        raise ValueError(
            f"Expected 2D image at {file_path}, got {data.shape}"
        )

    return data


def load_camus_frame(patient_id, phase):

    patient_dir = DATASET_PATH / patient_id

    image_path = (
        patient_dir /
        f"{patient_id}_4CH_{phase}.nii.gz"
    )

    mask_path = (
        patient_dir /
        f"{patient_id}_4CH_{phase}_gt.nii.gz"
    )

    image = load_nifti_2d(image_path)
    full_mask = load_nifti_2d(mask_path)

    # CAMUS label 1 = LV cavity
    lv_mask = (full_mask == 1).astype(np.float32)

    return image, lv_mask


def preprocess_frame(image, mask, resolution=SELECTED_RESOLUTION):

    image_resized = cv2.resize(
        image,
        (resolution, resolution),
        interpolation=cv2.INTER_LINEAR
    )

    mask_resized = cv2.resize(
        mask,
        (resolution, resolution),
        interpolation=cv2.INTER_NEAREST
    )

    # Per-image min-max normalization
    minimum = float(image_resized.min())
    maximum = float(image_resized.max())

    if maximum > minimum:
        image_resized = (
            (image_resized - minimum) /
            (maximum - minimum)
        )
    else:
        image_resized = np.zeros_like(
            image_resized,
            dtype=np.float32
        )

    image_resized = image_resized.astype(np.float32)
    mask_resized = (mask_resized > 0.5).astype(np.float32)

    return (
        image_resized[..., None],
        mask_resized[..., None]
    )


print("✓ CAMUS loading and preprocessing functions ready.")

✓ CAMUS loading and preprocessing functions ready.


In [ ]:
# ================================================================
# PHASE 1B — CELL 10
# MEMORY-FRIENDLY DATA GENERATOR
# ================================================================

def frame_generator(patient_ids, shuffle=True, seed=SEED):

    frame_list = [
        (pid, phase)
        for pid in patient_ids
        for phase in ["ED", "ES"]
    ]

    rng = np.random.default_rng(seed)

    while True:

        indices = np.arange(len(frame_list))

        if shuffle:
            rng.shuffle(indices)

        for index in indices:

            pid, phase = frame_list[index]

            image, mask = load_camus_frame(
                pid,
                phase
            )

            x, y = preprocess_frame(
                image,
                mask,
                SELECTED_RESOLUTION
            )

            yield x, y

            del image, mask, x, y

In [ ]:
# ================================================================
# PHASE 1B — CELL 11
# TF.DATA DATASET CREATION
# ================================================================

BATCH_SIZE = 4

TRAIN_FRAMES = len(train_ids) * 2
VAL_FRAMES = len(val_ids) * 2

TRAIN_STEPS = TRAIN_FRAMES // BATCH_SIZE
VAL_STEPS = VAL_FRAMES // BATCH_SIZE


output_signature = (
    tf.TensorSpec(
        shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        ),
        dtype=tf.float32
    ),
    tf.TensorSpec(
        shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        ),
        dtype=tf.float32
    )
)


train_dataset = tf.data.Dataset.from_generator(
    lambda: frame_generator(
        train_ids,
        shuffle=True,
        seed=SEED
    ),
    output_signature=output_signature
).batch(
    BATCH_SIZE,
    drop_remainder=True
).prefetch(1)


val_dataset = tf.data.Dataset.from_generator(
    lambda: frame_generator(
        val_ids,
        shuffle=False,
        seed=SEED
    ),
    output_signature=output_signature
).batch(
    BATCH_SIZE,
    drop_remainder=False
).prefetch(1)


print("Training frames:", TRAIN_FRAMES)
print("Validation frames:", VAL_FRAMES)

print("Batch size:", BATCH_SIZE)
print("Training steps:", TRAIN_STEPS)
print("Validation steps:", VAL_STEPS)

print("\n✓ Memory-friendly datasets ready.")

Training frames: 800
Validation frames: 100
Batch size: 4
Training steps: 200
Validation steps: 25

✓ Memory-friendly datasets ready.


In [ ]:
# ================================================================
# PHASE 1B — CELL 12
# LOSS + TRAINING METRICS
# ================================================================

SMOOTH = 1e-6


def dice_coefficient(y_true, y_pred):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    denominator = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3]
        )
    )

    dice = (
        (2.0 * intersection + SMOOTH)
        /
        (denominator + SMOOTH)
    )

    return tf.reduce_mean(dice)


def iou_metric(y_true, y_pred):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    union = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3]
        )
        -
        intersection
    )

    iou = (
        (intersection + SMOOTH)
        /
        (union + SMOOTH)
    )

    return tf.reduce_mean(iou)


def dice_loss(y_true, y_pred):

    return 1.0 - dice_coefficient(
        y_true,
        y_pred
    )


def bce_dice_loss(y_true, y_pred):

    bce = tf.keras.losses.binary_crossentropy(
        y_true,
        y_pred
    )

    bce = tf.reduce_mean(bce)

    return bce + dice_loss(
        y_true,
        y_pred
    )


print("✓ BCE + Dice loss ready.")
print("✓ Dice and IoU metrics ready.")

✓ BCE + Dice loss ready.
✓ Dice and IoU metrics ready.


In [ ]:
# ================================================================
# PHASE 1B — CELL 13
# FIXED BASELINE U-NET
# ================================================================

def conv_block(x, filters):

    x = tf.keras.layers.Conv2D(
        filters,
        3,
        activation="relu",
        padding="same"
    )(x)

    x = tf.keras.layers.Conv2D(
        filters,
        3,
        activation="relu",
        padding="same"
    )(x)

    return x


def build_baseline_unet(
    input_shape=(256, 256, 1)
):

    inputs = tf.keras.Input(
        shape=input_shape
    )

    # Encoder
    c1 = conv_block(inputs, 16)
    p1 = tf.keras.layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 32)
    p2 = tf.keras.layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 64)
    p3 = tf.keras.layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 128)
    p4 = tf.keras.layers.MaxPooling2D()(c4)

    # Bottleneck
    c5 = conv_block(p4, 256)

    # Decoder
    u6 = tf.keras.layers.Conv2DTranspose(
        128, 2, strides=2, padding="same"
    )(c5)

    u6 = tf.keras.layers.Concatenate()(
        [u6, c4]
    )

    c6 = conv_block(u6, 128)

    u7 = tf.keras.layers.Conv2DTranspose(
        64, 2, strides=2, padding="same"
    )(c6)

    u7 = tf.keras.layers.Concatenate()(
        [u7, c3]
    )

    c7 = conv_block(u7, 64)

    u8 = tf.keras.layers.Conv2DTranspose(
        32, 2, strides=2, padding="same"
    )(c7)

    u8 = tf.keras.layers.Concatenate()(
        [u8, c2]
    )

    c8 = conv_block(u8, 32)

    u9 = tf.keras.layers.Conv2DTranspose(
        16, 2, strides=2, padding="same"
    )(c8)

    u9 = tf.keras.layers.Concatenate()(
        [u9, c1]
    )

    c9 = conv_block(u9, 16)

    outputs = tf.keras.layers.Conv2D(
        1,
        1,
        activation="sigmoid"
    )(c9)

    segmentation_model = tf.keras.Model(
        inputs,
        outputs,
        name="Baseline_UNet"
    )

    return segmentation_model

In [ ]:
# ================================================================
# PHASE 1B — CELL 14
# ARCHITECTURE VERIFICATION
# ================================================================

tf.keras.backend.clear_session()
gc.collect()

architecture_check_model = build_baseline_unet(
    input_shape=(
        SELECTED_RESOLUTION,
        SELECTED_RESOLUTION,
        1
    )
)

BASELINE_PARAMETER_COUNT = (
    architecture_check_model.count_params()
)

print(
    "Baseline U-Net parameters:",
    BASELINE_PARAMETER_COUNT
)

assert BASELINE_PARAMETER_COUNT == 1940817, (
    "Architecture does not match Phase 1A!"
)

print("✓ Architecture exactly matches Phase 1A.")

del architecture_check_model
tf.keras.backend.clear_session()
gc.collect()

Baseline U-Net parameters: 1940817
✓ Architecture exactly matches Phase 1A.


0

In [ ]:
# ================================================================
# PHASE 1B — CELL 15
# CREATE IDENTICAL INITIAL WEIGHTS
# ================================================================

INITIAL_WEIGHTS_PATH = (
    P1B_CONFIG_DIR
    / "baseline_initial_weights.weights.h5"
)

tf.keras.backend.clear_session()
gc.collect()

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

initialization_model = build_baseline_unet(
    input_shape=(
        SELECTED_RESOLUTION,
        SELECTED_RESOLUTION,
        1
    )
)

initialization_model.save_weights(
    INITIAL_WEIGHTS_PATH
)

print("Initial weights saved:")
print(INITIAL_WEIGHTS_PATH)

print(
    "\nParameters:",
    initialization_model.count_params()
)

del initialization_model

tf.keras.backend.clear_session()
gc.collect()

print("\n✓ One common initialization created for all LR runs.")

Initial weights saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/configs/baseline_initial_weights.weights.h5

Parameters: 1940817

✓ One common initialization created for all LR runs.


In [ ]:
# ================================================================
# PHASE 1B — CELL 16
# SAVE LEARNING-RATE EXPERIMENT PROTOCOL
# ================================================================

LEARNING_RATES = [
    1e-3,
    1e-4,
    1e-5
]

MAX_EPOCHS = 30


lr_protocol = {

    "experiment":
        "Phase 1B-1 — Learning Rate",

    "objective":
        "Compare learning rates while keeping all other "
        "training and model settings fixed.",

    "resolution":
        SELECTED_RESOLUTION,

    "architecture":
        "Baseline U-Net",

    "parameters":
        BASELINE_PARAMETER_COUNT,

    "learning_rates":
        LEARNING_RATES,

    "optimizer":
        "Adam",

    "batch_size":
        BATCH_SIZE,

    "loss":
        "BCE + Dice",

    "augmentation":
        False,

    "maximum_epochs":
        MAX_EPOCHS,

    "checkpoint_monitor":
        "val_dice_coefficient",

    "checkpoint_mode":
        "max",

    "threshold":
        0.50,

    "seed":
        SEED,

    "train_patients":
        len(train_ids),

    "validation_patients":
        len(val_ids),

    "test_used":
        False,

    "common_initialization":
        str(INITIAL_WEIGHTS_PATH)
}


LR_PROTOCOL_PATH = (
    P1B_CONFIG_DIR
    / "learning_rate_experiment_protocol.json"
)


with open(
    LR_PROTOCOL_PATH,
    "w"
) as f:

    json.dump(
        lr_protocol,
        f,
        indent=2
    )


print("✓ LR protocol saved:")
print(LR_PROTOCOL_PATH)

✓ LR protocol saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/configs/learning_rate_experiment_protocol.json


In [ ]:
# ================================================================
# PHASE 1B — CELL 17
# LEARNING-RATE TRAINING FUNCTION
# ================================================================

def lr_name(learning_rate):

    mapping = {
        1e-3: "1e-3",
        1e-4: "1e-4",
        1e-5: "1e-5"
    }

    return mapping[learning_rate]


def train_learning_rate_experiment(
    learning_rate
):

    run_name = lr_name(
        learning_rate
    )

    print("\n" + "=" * 70)
    print(
        f"TRAINING LEARNING RATE = {run_name}"
    )
    print("=" * 70)

    # ------------------------------------------------------------
    # Clean previous model from GPU/RAM
    # ------------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    # ------------------------------------------------------------
    # Re-establish reproducibility
    # ------------------------------------------------------------

    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    # ------------------------------------------------------------
    # Build fresh model
    # ------------------------------------------------------------

    segmentation_model = build_baseline_unet(
        input_shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        )
    )

    # ------------------------------------------------------------
    # Load IDENTICAL initial weights
    # ------------------------------------------------------------

    segmentation_model.load_weights(
        INITIAL_WEIGHTS_PATH
    )

    # ------------------------------------------------------------
    # Compile
    # ------------------------------------------------------------

    segmentation_model.compile(

        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),

        loss=bce_dice_loss,

        metrics=[
            dice_coefficient,
            iou_metric
        ]
    )

    # ------------------------------------------------------------
    # Output paths
    # ------------------------------------------------------------

    model_path = (
        P1B_MODEL_DIR
        / f"lr_{run_name}_best.keras"
    )

    history_path = (
        P1B_HISTORY_DIR
        / f"lr_{run_name}_history.csv"
    )

    # ------------------------------------------------------------
    # Best validation-Dice checkpoint
    # ------------------------------------------------------------

    checkpoint = tf.keras.callbacks.ModelCheckpoint(

        filepath=str(model_path),

        monitor="val_dice_coefficient",

        mode="max",

        save_best_only=True,

        verbose=1
    )

    # ------------------------------------------------------------
    # Train
    # ------------------------------------------------------------

    start_time = time.time()

    history = segmentation_model.fit(

        train_dataset,

        validation_data=val_dataset,

        epochs=MAX_EPOCHS,

        steps_per_epoch=TRAIN_STEPS,

        validation_steps=VAL_STEPS,

        callbacks=[
            checkpoint
        ],

        verbose=1
    )

    training_minutes = (
        time.time() - start_time
    ) / 60.0

    # ------------------------------------------------------------
    # Save history
    # ------------------------------------------------------------

    history_df = pd.DataFrame(
        history.history
    )

    history_df.index = (
        np.arange(len(history_df)) + 1
    )

    history_df.index.name = "epoch"

    history_df.to_csv(
        history_path
    )

    # ------------------------------------------------------------
    # Determine best epoch
    # ------------------------------------------------------------

    val_dice_values = np.asarray(
        history.history[
            "val_dice_coefficient"
        ],
        dtype=float
    )

    best_index = int(
        np.argmax(val_dice_values)
    )

    best_epoch = best_index + 1

    best_val_dice = float(
        val_dice_values[best_index]
    )

    best_val_iou = float(
        history.history[
            "val_iou_metric"
        ][best_index]
    )

    result = {

        "learning_rate":
            learning_rate,

        "learning_rate_name":
            run_name,

        "parameters":
            int(
                segmentation_model.count_params()
            ),

        "epochs_completed":
            len(history.history["loss"]),

        "best_epoch":
            best_epoch,

        "best_val_dice_training":
            best_val_dice,

        "val_iou_at_best_dice_epoch":
            best_val_iou,

        "training_minutes":
            training_minutes,

        "model_path":
            str(model_path),

        "history_path":
            str(history_path)
    }

    print("\n" + "-" * 70)

    print(
        f"LR {run_name} COMPLETE"
    )

    print(
        "Best epoch:",
        best_epoch
    )

    print(
        "Best validation Dice:",
        round(best_val_dice, 6)
    )

    print(
        "Validation IoU at best epoch:",
        round(best_val_iou, 6)
    )

    print(
        "Training time:",
        round(training_minutes, 2),
        "minutes"
    )

    print(
        "Best model:",
        model_path
    )

    print("-" * 70)

    # ------------------------------------------------------------
    # Clean GPU memory
    # ------------------------------------------------------------

    del segmentation_model
    del history

    tf.keras.backend.clear_session()
    gc.collect()

    return result

In [ ]:
# ================================================================
# PHASE 1B — CELL 18
# RUN 1: LEARNING RATE = 1e-3
# ================================================================

lr_result_1e3 = train_learning_rate_experiment(
    1e-3
)

lr_result_1e3


TRAINING LEARNING RATE = 1e-3
Epoch 1/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - dice_coefficient: 0.3728 - iou_metric: 0.2557 - loss: 0.9362
Epoch 1: val_dice_coefficient improved from None to 0.74080, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/lr_1e-3_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/lr_1e-3_best.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 30s 111ms/step - dice_coefficient: 0.5399 - iou_metric: 0.3956 - loss: 0.6667 - val_dice_coefficient: 0.7408 - val_iou_metric: 0.5930 - val_loss: 0.3541
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - dice_coefficient: 0.7624 - iou_metric: 0.6263 - loss: 0.3487
Epoch 2: val_dice_coefficient improved from 0.74080 to 0.84366, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/lr_1e-3_best.keras

Epoch 2: finished saving model to /content/driv

{'learning_rate': 0.001,
 'learning_rate_name': '1e-3',
 'parameters': 1940817,
 'epochs_completed': 30,
 'best_epoch': 30,
 'best_val_dice_training': 0.9260080456733704,
 'val_iou_at_best_dice_epoch': 0.8650317192077637,
 'training_minutes': 10.241162570317586,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/lr_1e-3_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/histories/lr_1e-3_history.csv'}

In [ ]:
# ================================================================
# PHASE 1B — CELL 19
# RUN 2: LEARNING RATE = 1e-4
# ================================================================

lr_result_1e4 = train_learning_rate_experiment(
    1e-4
)

lr_result_1e4


TRAINING LEARNING RATE = 1e-4
Epoch 1/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - dice_coefficient: 0.1832 - iou_metric: 0.1053 - loss: 1.3305
Epoch 1: val_dice_coefficient improved from None to 0.43977, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/lr_1e-4_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/lr_1e-4_best.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 27s 105ms/step - dice_coefficient: 0.2759 - iou_metric: 0.1687 - loss: 1.0598 - val_dice_coefficient: 0.4398 - val_iou_metric: 0.2837 - val_loss: 0.7019
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - dice_coefficient: 0.5634 - iou_metric: 0.4030 - loss: 0.6001
Epoch 2: val_dice_coefficient improved from 0.43977 to 0.72617, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/lr_1e-4_best.keras

Epoch 2: finished saving model to /content/driv

{'learning_rate': 0.0001,
 'learning_rate_name': '1e-4',
 'parameters': 1940817,
 'epochs_completed': 30,
 'best_epoch': 30,
 'best_val_dice_training': 0.9254956245422363,
 'val_iou_at_best_dice_epoch': 0.8635912537574768,
 'training_minutes': 12.20138973792394,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/lr_1e-4_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/histories/lr_1e-4_history.csv'}

In [ ]:
# ================================================================
# PHASE 1B — CELL 20
# RUN 3: LEARNING RATE = 1e-5
# ================================================================

lr_result_1e5 = train_learning_rate_experiment(
    1e-5
)

lr_result_1e5


TRAINING LEARNING RATE = 1e-5
Epoch 1/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - dice_coefficient: 0.1527 - iou_metric: 0.0836 - loss: 1.5333
Epoch 1: val_dice_coefficient improved from None to 0.14411, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/lr_1e-5_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/lr_1e-5_best.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 29s 104ms/step - dice_coefficient: 0.1520 - iou_metric: 0.0832 - loss: 1.5304 - val_dice_coefficient: 0.1441 - val_iou_metric: 0.0784 - val_loss: 1.5262
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - dice_coefficient: 0.1200 - iou_metric: 0.0646 - loss: 1.4339
Epoch 2: val_dice_coefficient did not improve from 0.14411
200/200 ━━━━━━━━━━━━━━━━━━━━ 20s 101ms/step - dice_coefficient: 0.1053 - iou_metric: 0.0561 - loss: 1.3584 - val_dice_coefficient: 0.1311 - val_iou_metric: 0.0706 - val_loss:

{'learning_rate': 1e-05,
 'learning_rate_name': '1e-5',
 'parameters': 1940817,
 'epochs_completed': 30,
 'best_epoch': 30,
 'best_val_dice_training': 0.8688406348228455,
 'val_iou_at_best_dice_epoch': 0.7717862725257874,
 'training_minutes': 11.432759237289428,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/models/lr_1e-5_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/histories/lr_1e-5_history.csv'}

In [ ]:
# ================================================================
# PHASE 1B — CELL 21
# SAVE LEARNING-RATE TRAINING SUMMARY
# ================================================================

lr_training_summary = pd.DataFrame([
    lr_result_1e3,
    lr_result_1e4,
    lr_result_1e5
])

LR_TRAINING_SUMMARY_PATH = (
    P1B_TABLE_DIR
    / "learning_rate_training_summary.csv"
)

lr_training_summary.to_csv(
    LR_TRAINING_SUMMARY_PATH,
    index=False
)

display(
    lr_training_summary[
        [
            "learning_rate_name",
            "parameters",
            "best_epoch",
            "best_val_dice_training",
            "val_iou_at_best_dice_epoch",
            "training_minutes"
        ]
    ]
)

print("\n✓ Training summary saved:")
print(LR_TRAINING_SUMMARY_PATH)

,learning_rate_name,parameters,best_epoch,best_val_dice_training,val_iou_at_best_dice_epoch,training_minutes
0,1e-3,1940817,30,0.926008,0.865032,10.241163
1,1e-4,1940817,30,0.925496,0.863591,12.201390
2,1e-5,1940817,30,0.868841,0.771786,11.432759



✓ Training summary saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/learning_rate_training_summary.csv


In [ ]:
# ================================================================
# PHASE 1B — CELL 22
# POSTPROCESSING + VALIDATION METRICS
# ================================================================

from scipy.ndimage import binary_fill_holes
from scipy.spatial.distance import cdist


def keep_largest_component(mask):

    mask = np.asarray(mask).astype(np.uint8)

    if mask.sum() == 0:
        return np.zeros_like(mask, dtype=bool)

    n_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mask,
            connectivity=8
        )
    )

    if n_labels <= 1:
        return np.zeros_like(mask, dtype=bool)

    largest_label = (
        1 +
        np.argmax(
            stats[1:, cv2.CC_STAT_AREA]
        )
    )

    return labels == largest_label


def postprocess_prediction(mask):

    largest = keep_largest_component(mask)

    filled = binary_fill_holes(
        largest
    )

    return np.asarray(
        filled,
        dtype=bool
    )


def binary_dice(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    intersection = np.logical_and(
        gt, pred
    ).sum()

    denominator = gt.sum() + pred.sum()

    if denominator == 0:
        return 1.0

    return (
        2.0 * intersection
        / denominator
    )


def binary_iou(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    intersection = np.logical_and(
        gt, pred
    ).sum()

    union = np.logical_or(
        gt, pred
    ).sum()

    if union == 0:
        return 1.0

    return intersection / union


def binary_precision(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    tp = np.logical_and(gt, pred).sum()
    fp = np.logical_and(~gt, pred).sum()

    if (tp + fp) == 0:
        return 1.0 if gt.sum() == 0 else 0.0

    return tp / (tp + fp)


def binary_recall(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    tp = np.logical_and(gt, pred).sum()
    fn = np.logical_and(gt, ~pred).sum()

    if (tp + fn) == 0:
        return 1.0

    return tp / (tp + fn)


def boundary_points(mask):

    mask = np.asarray(mask).astype(np.uint8)

    kernel = np.ones(
        (3, 3),
        dtype=np.uint8
    )

    eroded = cv2.erode(
        mask,
        kernel,
        iterations=1
    )

    boundary = (
        mask.astype(bool)
        &
        ~eroded.astype(bool)
    )

    return np.column_stack(
        np.where(boundary)
    )


def hd95_pixels(gt, pred):

    gt_points = boundary_points(gt)
    pred_points = boundary_points(pred)

    if (
        len(gt_points) == 0
        or len(pred_points) == 0
    ):
        return np.nan

    distances = cdist(
        gt_points,
        pred_points
    )

    gt_to_pred = distances.min(axis=1)
    pred_to_gt = distances.min(axis=0)

    all_distances = np.concatenate([
        gt_to_pred,
        pred_to_gt
    ])

    return float(
        np.percentile(
            all_distances,
            95
        )
    )


print("✓ Postprocessing and evaluation metrics ready.")

✓ Postprocessing and evaluation metrics ready.


In [ ]:
# ================================================================
# PHASE 1B — CELL 23
# CHECKPOINT EVALUATION FUNCTION
# ================================================================

THRESHOLD = 0.50


def evaluate_lr_checkpoint(
    learning_rate
):

    run_name = lr_name(
        learning_rate
    )

    model_path = (
        P1B_MODEL_DIR
        / f"lr_{run_name}_best.keras"
    )

    assert model_path.exists(), (
        f"Checkpoint missing: {model_path}"
    )

    print("\n" + "=" * 70)
    print(
        f"EVALUATING LR = {run_name}"
    )
    print("=" * 70)

    tf.keras.backend.clear_session()
    gc.collect()

    evaluation_model = tf.keras.models.load_model(
        model_path,
        custom_objects={
            "bce_dice_loss": bce_dice_loss,
            "dice_coefficient": dice_coefficient,
            "iou_metric": iou_metric
        }
    )

    records = []

    for patient_number, pid in enumerate(
        val_ids,
        start=1
    ):

        for phase in ["ED", "ES"]:

            image, gt_mask = load_camus_frame(
                pid,
                phase
            )

            x, gt_resized = preprocess_frame(
                image,
                gt_mask,
                SELECTED_RESOLUTION
            )

            probability = evaluation_model(
                x[None, ...],
                training=False
            ).numpy()[0, :, :, 0]

            pred_raw = (
                probability >= THRESHOLD
            )

            pred_mask = postprocess_prediction(
                pred_raw
            )

            gt_binary = (
                gt_resized[:, :, 0] > 0.5
            )

            gt_area = int(
                gt_binary.sum()
            )

            pred_area = int(
                pred_mask.sum()
            )

            if gt_area > 0:

                signed_area_error_pct = (
                    (pred_area - gt_area)
                    / gt_area
                    * 100.0
                )

                abs_area_error_pct = abs(
                    signed_area_error_pct
                )

            else:

                signed_area_error_pct = np.nan
                abs_area_error_pct = np.nan

            hd95 = hd95_pixels(
                gt_binary,
                pred_mask
            )

            records.append({

                "patient_id":
                    pid,

                "phase":
                    phase,

                "learning_rate":
                    learning_rate,

                "learning_rate_name":
                    run_name,

                "dice":
                    binary_dice(
                        gt_binary,
                        pred_mask
                    ),

                "iou":
                    binary_iou(
                        gt_binary,
                        pred_mask
                    ),

                "precision":
                    binary_precision(
                        gt_binary,
                        pred_mask
                    ),

                "recall":
                    binary_recall(
                        gt_binary,
                        pred_mask
                    ),

                "hd95_pixels":
                    hd95,

                "normalized_hd95":
                    (
                        hd95
                        / SELECTED_RESOLUTION
                        if np.isfinite(hd95)
                        else np.nan
                    ),

                "gt_area_pixels":
                    gt_area,

                "pred_area_pixels":
                    pred_area,

                "signed_area_error_pct":
                    signed_area_error_pct,

                "abs_area_error_pct":
                    abs_area_error_pct
            })

            del (
                image,
                gt_mask,
                x,
                gt_resized,
                probability,
                pred_raw,
                pred_mask,
                gt_binary
            )

        if patient_number % 10 == 0:
            print(
                f"Evaluated "
                f"{patient_number}/{len(val_ids)} "
                f"patients"
            )

    results_df = pd.DataFrame(
        records
    )

    output_path = (
        P1B_TABLE_DIR
        / f"lr_{run_name}_validation_metrics.csv"
    )

    results_df.to_csv(
        output_path,
        index=False
    )

    print("\nFrames evaluated:",
          len(results_df))

    print("Saved:")
    print(output_path)

    del evaluation_model

    tf.keras.backend.clear_session()
    gc.collect()

    return results_df

In [ ]:
# ================================================================
# PHASE 1B — CELL 24
# EVALUATE ALL LEARNING RATES
# ================================================================

lr_eval_1e3 = evaluate_lr_checkpoint(
    1e-3
)

lr_eval_1e4 = evaluate_lr_checkpoint(
    1e-4
)

lr_eval_1e5 = evaluate_lr_checkpoint(
    1e-5
)

print("\n✓ All LR checkpoints evaluated.")


EVALUATING LR = 1e-3
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/lr_1e-3_validation_metrics.csv

EVALUATING LR = 1e-4
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/lr_1e-4_validation_metrics.csv

EVALUATING LR = 1e-5
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/lr_1e-5_validation_metrics.csv

✓ All LR checkpoints evaluated.


In [ ]:
# ================================================================
# PHASE 1B — CELL 25
# LEARNING-RATE VALIDATION COMPARISON
# ================================================================

def summarize_lr_results(
    evaluation_df,
    training_result
):

    ed = evaluation_df[
        evaluation_df["phase"] == "ED"
    ]

    es = evaluation_df[
        evaluation_df["phase"] == "ES"
    ]

    return {

        "learning_rate_name":
            training_result[
                "learning_rate_name"
            ],

        "learning_rate":
            training_result[
                "learning_rate"
            ],

        "parameters":
            training_result[
                "parameters"
            ],

        "best_epoch":
            training_result[
                "best_epoch"
            ],

        "training_minutes":
            training_result[
                "training_minutes"
            ],

        "overall_dice":
            evaluation_df["dice"].mean(),

        "ED_dice":
            ed["dice"].mean(),

        "ES_dice":
            es["dice"].mean(),

        "overall_iou":
            evaluation_df["iou"].mean(),

        "precision":
            evaluation_df[
                "precision"
            ].mean(),

        "recall":
            evaluation_df[
                "recall"
            ].mean(),

        "normalized_hd95":
            evaluation_df[
                "normalized_hd95"
            ].mean(),

        "ED_area_MAE_pct":
            ed[
                "abs_area_error_pct"
            ].mean(),

        "ES_area_MAE_pct":
            es[
                "abs_area_error_pct"
            ].mean(),

        "overall_area_MAE_pct":
            evaluation_df[
                "abs_area_error_pct"
            ].mean()
    }


lr_comparison = pd.DataFrame([

    summarize_lr_results(
        lr_eval_1e3,
        lr_result_1e3
    ),

    summarize_lr_results(
        lr_eval_1e4,
        lr_result_1e4
    ),

    summarize_lr_results(
        lr_eval_1e5,
        lr_result_1e5
    )

])


LR_COMPARISON_PATH = (
    P1B_TABLE_DIR
    / "learning_rate_validation_comparison.csv"
)

lr_comparison.to_csv(
    LR_COMPARISON_PATH,
    index=False
)


display(
    lr_comparison.round(4)
)

print("\n✓ Final LR comparison saved:")
print(LR_COMPARISON_PATH)

,learning_rate_name,learning_rate,parameters,best_epoch,training_minutes,overall_dice,ED_dice,ES_dice,overall_iou,precision,recall,normalized_hd95,ED_area_MAE_pct,ES_area_MAE_pct,overall_area_MAE_pct
0,1e-3,0.0010,1940817,30,10.2412,0.9280,0.9374,0.9186,0.8685,0.9253,0.9372,0.0244,7.5395,10.6126,9.0761
1,1e-4,0.0001,1940817,30,12.2014,0.9295,0.9401,0.9188,0.8705,0.9312,0.9334,0.0234,6.4293,10.0577,8.2435
2,1e-5,0.0000,1940817,30,11.4328,0.8828,0.8978,0.8678,0.7943,0.8819,0.8997,0.0397,11.3755,20.3970,15.8862



✓ Final LR comparison saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/tables/learning_rate_validation_comparison.csv


In [ ]:
# ================================================================
# PHASE 1B — CELL 26
# FINALIZE LEARNING-RATE SELECTION
# ================================================================

SELECTED_LEARNING_RATE = 1e-4

selected_row = lr_comparison[
    lr_comparison["learning_rate"] == SELECTED_LEARNING_RATE
].iloc[0]


lr_decision = {

    "experiment":
        "Phase 1B-1 — Controlled Learning Rate Experiment",

    "status":
        "COMPLETED",

    "selected_learning_rate":
        SELECTED_LEARNING_RATE,

    "learning_rates_compared":
        [1e-3, 1e-4, 1e-5],

    "fixed_resolution":
        SELECTED_RESOLUTION,

    "fixed_architecture":
        "Baseline U-Net",

    "parameter_count":
        BASELINE_PARAMETER_COUNT,

    "fixed_optimizer":
        "Adam",

    "fixed_batch_size":
        BATCH_SIZE,

    "fixed_loss":
        "BCE + Dice",

    "augmentation":
        False,

    "epochs":
        MAX_EPOCHS,

    "threshold":
        THRESHOLD,

    "selection_dataset":
        "Validation set only",

    "test_set_used":
        False,

    "selected_validation_results": {

        "overall_dice":
            float(selected_row["overall_dice"]),

        "ED_dice":
            float(selected_row["ED_dice"]),

        "ES_dice":
            float(selected_row["ES_dice"]),

        "overall_iou":
            float(selected_row["overall_iou"]),

        "precision":
            float(selected_row["precision"]),

        "recall":
            float(selected_row["recall"]),

        "normalized_hd95":
            float(selected_row["normalized_hd95"]),

        "ED_area_MAE_pct":
            float(selected_row["ED_area_MAE_pct"]),

        "ES_area_MAE_pct":
            float(selected_row["ES_area_MAE_pct"]),

        "overall_area_MAE_pct":
            float(selected_row["overall_area_MAE_pct"])
    },

    "selection_reason": (
        "Among the tested learning rates, 1e-4 provided the strongest "
        "overall validation performance. It achieved higher overall Dice "
        "and IoU than 1e-3 and 1e-5, while also producing the lowest "
        "normalized HD95 and the lowest ED, ES, and overall LV area MAE. "
        "Because downstream quantitative LV analysis depends on both "
        "segmentation overlap and geometric fidelity, 1e-4 was selected "
        "for subsequent controlled experiments."
    ),

    "next_experiment":
        "Phase 1B-2 — Optimizer Comparison"
}


LR_DECISION_PATH = (
    P1B_CONFIG_DIR
    / "phase1B1_learning_rate_decision.json"
)


with open(
    LR_DECISION_PATH,
    "w"
) as f:

    json.dump(
        lr_decision,
        f,
        indent=2
    )


print("=" * 70)
print("PHASE 1B-1 COMPLETED")
print("=" * 70)

print("\nSelected learning rate:")
print(SELECTED_LEARNING_RATE)

print("\nLocked settings:")
print(f"Resolution     : {SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}")
print(f"Learning rate  : {SELECTED_LEARNING_RATE}")

print("\nSelected validation performance:")
print(
    f"Overall Dice   : {selected_row['overall_dice']:.4f}"
)
print(
    f"ED Dice        : {selected_row['ED_dice']:.4f}"
)
print(
    f"ES Dice        : {selected_row['ES_dice']:.4f}"
)
print(
    f"Overall IoU    : {selected_row['overall_iou']:.4f}"
)
print(
    f"ED Area MAE    : {selected_row['ED_area_MAE_pct']:.2f}%"
)
print(
    f"ES Area MAE    : {selected_row['ES_area_MAE_pct']:.2f}%"
)
print(
    f"Overall Area MAE: {selected_row['overall_area_MAE_pct']:.2f}%"
)

print("\nTest set used: NO")

print("\nNext:")
print("Phase 1B-2 — Optimizer Comparison")

print("\nDecision saved:")
print(LR_DECISION_PATH)

print("\n✓ Learning rate officially locked at 1e-4.")

PHASE 1B-1 COMPLETED

Selected learning rate:
0.0001

Locked settings:
Resolution     : 256×256
Learning rate  : 0.0001

Selected validation performance:
Overall Dice   : 0.9295
ED Dice        : 0.9401
ES Dice        : 0.9188
Overall IoU    : 0.8705
ED Area MAE    : 6.43%
ES Area MAE    : 10.06%
Overall Area MAE: 8.24%

Test set used: NO

Next:
Phase 1B-2 — Optimizer Comparison

Decision saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1B_hyperparameters/configs/phase1B1_learning_rate_decision.json

✓ Learning rate officially locked at 1e-4.
